1. ask_agent(pipe)를 단독으로 2~3회 호출해 반환값이 252,000 근처 float인지, __CODE_ERROR__나 __TIMEOUT__이 아닌지 확인합니다. 이 단계에서 실패하면 원인은 대개 competition/CardBase/all.parquet 경로 또는 AWS 자격증명이니, 두 번째 반환값(raw)을 그대로 출력해 확인하세요.

2. temperature 설정 경로를 고칩니다. pipe.main_pipeline.answerer.temperature = 0.5와 함께 pipe.main_pipeline.answerer.top_p = 1.0으로 두고, 오류 수정 파이프라인도 pipe.error_fix_pipeline.answerer에 같은 값을 적용해 조건을 일치시킵니다. 이 수정은 dp_agent_wrap.py의 build_pipe 안에 넣는 게 낫습니다.

3. 노트북 첫 셀에서 core.utils, from main import ..., boto3, load_dotenv, prepare_dataset 임포트를 제거하고 pandas, numpy, scipy.stats와 래퍼에서 쓰는 것만 남깁니다.

4. run_fixed_k_agent를 k=5, seed=0으로 한 번 돌려 pre_unique(1이면 변동성 없음), err_pre, fail을 확인합니다. temperature 조정이 실제로 변동성을 만드는지 여기서 판정됩니다.

5. raws를 결과 dict에서 빼내 별도 리스트나 파일로 저장하도록 바꿉니다. DataFrame으로 라운드를 쌓을 때 열이 오염되는 걸 막기 위함입니다.

6. 비용 구조를 캐시 방식으로 전환합니다. 에이전트 응답을 M개(예: 50) 한 번 수집해 저장하고, run_fixed_k_agent가 LLM을 다시 부르지 않고 그 풀에서 k개를 복원추출한 뒤 라플라스를 더하도록 분리합니다. temperature 0 조건과 0.5 조건의 풀을 각각 만들어 두면 두 조건 비교도 같은 비용으로 됩니다.

7. 캐시 풀이 준비된 뒤 기존 explore 루프의 run_fixed_k 호출부를 캐시 기반 함수로 교체하고, 출력에 err_pre를 추가해 커버리지 하락의 원인이 노이즈인지 에이전트 편향인지 구분해 봅니다.

In [1]:
import os, numpy as np, pandas as pd, boto3
from pathlib import Path
from dotenv import load_dotenv
from scipy import stats

import core.utils as utils
from main import instantiate_pipeline_from_yaml

from dp_agent_wrap import prepare_dataset, build_pipe, ask_agent, DS_NAME, COL
from dp_agent_wrap import OpenAIAnswerer
from dp_agent_wrap import InlineStatementExecutor

# pipe = build_pipe(temperature=0.5)   # 변동성 관찰 조건

c:\Users\seude\miniconda3\envs\dp4agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


> 노트북이 이전에 로드한 모듈을 캐시하고 있는 경우 새로운 Class를 못 찾음. 리로드(아래 셀) 필요.

In [10]:
import dp_agent_wrap, importlib
print("OpenAIAnswerer" in open("dp_agent_wrap.py", encoding="utf-8").read())  # 파일 확인
importlib.reload(dp_agent_wrap)
from dp_agent_wrap import OpenAIAnswerer
from dp_agent_wrap import InlineStatementExecutor

True


In [2]:
def build_pipe(config="config/claude3.5-sonnet.yaml",
               model="gpt-4o-mini", temperature=0.0):
    load_dotenv()
    pipe, _ = instantiate_pipeline_from_yaml(
        config_path=config,
        exemplar_indices=[(17,140),(0,132),(28,286),(31,303),(4,246),
                          (24,175),(20,176),(8,141),(14,12)],
        annotations_filename="annotation/annotations_cot.json",
        client=None, debug=False, lite=False,
    )
    # (1) answerer: Bedrock Claude → OpenAI
    pipe.main_pipeline.answerer      = OpenAIAnswerer(model=model, temperature=temperature, max_gen_len=300)
    pipe.error_fix_pipeline.answerer = OpenAIAnswerer(model=model, temperature=temperature, max_gen_len=1000)
    # (2) executor: multiprocessing → thread (Windows 호환)
    pipe.main_pipeline.executor      = InlineStatementExecutor(utils.generic_load_table)
    pipe.error_fix_pipeline.executor = InlineStatementExecutor(utils.generic_load_table)
    return pipe

In [3]:
pipe = build_pipe(model="gpt-4o-mini", temperature=0.0)

val, raw = ask_agent(pipe)
print("value:", val)
print("---- raw ----")
print(raw)

Loading annotations from semeval train[:400]


value: 252614.0
---- raw ----
252614.0


동작했습니다. 오류 수정 루프도 돌지 않았으니(0th error fixing 없음) 코드 생성이 한 번에 성공한 것이고, OpenAI answerer + 스레드 실행기 조합이 정상입니다.

> 한 가지 확인이 필요합니다. 

TRUE(= df["Credit_Limit"].mean())와 252614.0을 비교해 보세요. 일치하면 에이전트가 정확히 평균을 계산한 것이고, 다르면 어떤 코드를 만들었는지 봐야 합니다. 그리고 값이 정수처럼 딱 떨어지는 게 걸립니다. 500행 평균이 소수점 없이 나오는 건 우연일 수도 있지만, 모델이 반올림했거나 round()를 넣었을 가능성이 있습니다. pipe.main_pipeline.run_one({"question": QUESTION, "dataset": DS_NAME})의 세 번째 반환값(postprocessed)을 찍으면 생성된 코드를 직접 볼 수 있습니다. 반올림이 들어갔다면 그 자체가 노이즈와 무관한 에이전트 편향이므로 err_pre에 잡히도록 남겨두면 됩니다.

* 다음 단계는 run_fixed_k_agent(pipe, D, eps=0.5, k=5, seed=0, true_mean=TRUE)입니다. LLM을 5번 부르니 pre_unique로 temperature 0에서의 변동성이 실제로 1인지 확인하세요.

In [15]:
print(type(pipe.main_pipeline.answerer))
print(type(pipe.error_fix_pipeline.answerer))
print(type(pipe.main_pipeline.executor))

<class 'core.model_calls.Claude_3_5_Sonnet_ModelAnswerer'>
<class 'core.model_calls.Claude_3_5_Sonnet_ModelAnswerer'>
<class 'dp_agent_wrap.InlineStatementExecutor'>


In [5]:
df   = pd.read_csv("CardBase.csv")
TRUE = df[COL].mean()
D    = (df[COL].max() - df[COL].min()) / len(df)

In [6]:
print(TRUE)

252614.0


In [ ]:
def run_fixed_k_agent(pipe, delta, eps, k, seed=0, true_mean=None):
    r = np.random.default_rng(seed)
    b = delta / eps

    pre, raws = [], []
    for _ in range(k):
        v, raw = ask_agent(pipe)
        pre.append(v); raws.append(raw)

    pre  = np.asarray(pre, dtype=float)
    ok   = np.isfinite(pre)
    v    = pre[ok] + r.laplace(0.0, b, size=ok.sum())   # 에이전트 출력에 노이즈
    n    = v.size

    if n < 2:
        return dict(k=k, n_valid=n, mean=np.nan, moe=np.nan, relw=np.nan,
                    ci_lo=np.nan, ci_hi=np.nan, covered=pd.NA,
                    resp_pre_mean=np.nan, err_pre=np.nan, fail=k-n)

    mean = v.mean()
    se   = v.std(ddof=1) / np.sqrt(n)
    moe  = stats.t.ppf(0.975, n-1) * se
    lo, hi = mean - moe, mean + moe

    return dict(
        k=k, n_valid=n, fail=k-n,
        mean=mean, moe=moe, relw=2*moe/abs(mean), ci_lo=lo, ci_hi=hi,
        covered=(lo <= true_mean <= hi) if true_mean is not None else pd.NA,
        resp_pre_mean=pre[ok].mean(),                    # 노이즈 전 평균
        err_pre=(pre[ok].mean() - true_mean) if true_mean is not None else np.nan,
        pre_unique=len(set(np.round(pre[ok], 6))),       # 에이전트 변동성 지표
        raws=raws,
    )